# 诊断 -> 引导 -> 学生记忆画像 Demo

这份 notebook 走一条完整但尽量轻的链路：

1. 直接读取 `wrong_binder` 里的现成例题与绑定结果
2. 用 `diagnosis_agent.py` 做错因诊断
3. 用 `coach_agent.py` 做几轮引导
4. 用 `student_memory_manager.py` 生成 `student_memory_profile`

这样你可以单独测试第四阶段，而不用先接前端。

In [1]:
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from coach_agent import FoundryCoachAgent, diagnose_environment
from diagnosis_agent import FoundryDiagnosisAgent, diagnosis_environment
from student_memory_manager import (
    build_profile,
    coach_response_to_memory_event,
    diagnosis_result_to_memory_event,
)

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'diagnosis_coach_memory_demo'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2))


## 1. 基本配置

- 题目和绑定结果直接读取 `wrong_question_binder_playground`
- 这里默认会真实调用模型
- 这套诊断 / 引导 agent 目前走的是 Azure 凭证，不是单独的 API key 模式

如果你本机已经能正常跑之前的 `diagnosis_agent` / `coach_agent`，这里就可以直接继续。

In [13]:
QUESTION_PATH = PROJECT_ROOT / 'scratch' / 'wrong_question_binder_playground' / 'sequence_geometric_shift_question.json'
BINDING_PATH = PROJECT_ROOT / 'scratch' / 'wrong_question_binder_playground' / 'sequence_geometric_shift_binding.json'

STUDENT_ID = 'demo_student'
STUDENT_PROFILE = '学生会从递推式入手，但经常不会主动构造中间量，证明题收尾也不太稳。'

USE_DEFAULT_CREDENTIAL = False
USE_AI_DIAGNOSER = True
USE_AI_REPLY_ANALYZER = True

DIAGNOSIS_MODEL = 'gpt-4o-mini'
COACH_MODEL = 'gpt-4o-mini'
COACH_MAX_TURNS = 4

BASE_TIME = datetime(2026, 6, 23, 19, 0, tzinfo=timezone(timedelta(hours=8)))


In [14]:
QUESTION_RECORD = json.loads(QUESTION_PATH.read_text(encoding='utf-8'))
BINDING_RECORD = json.loads(BINDING_PATH.read_text(encoding='utf-8'))

QUESTION = QUESTION_RECORD['question_payload']
QUESTION_ID = BINDING_RECORD['question_id']

BINDING_CONTEXT = {
    'primary_node_id': BINDING_RECORD['primary_binding']['node_id'],
    'secondary_node_ids': [item['node_id'] for item in BINDING_RECORD.get('secondary_bindings', [])],
    'linked_node_ids': [
        BINDING_RECORD['primary_binding']['node_id'],
        *[item['node_id'] for item in BINDING_RECORD.get('secondary_bindings', [])],
    ],
}

QUESTION_SUMMARY = {
    'question_id': QUESTION_ID,
    'stem': QUESTION['stem'],
    'student_answer': QUESTION['student_answer'],
    'source_name': QUESTION.get('source_name'),
    'source_section': QUESTION.get('source_section'),
    'binding_context': BINDING_CONTEXT,
}

show_json(QUESTION_SUMMARY)


{
  "question_id": "wq_72b68db02925",
  "stem": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
  "student_answer": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。",
  "source_name": "binder_notebook_demo",
  "source_section": "数列递推转等比",
  "binding_context": {
    "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
    "secondary_node_ids": [],
    "linked_node_ids": [
      "math.数列与不等式.数列.等差_等比数列.等比数列通项公式"
    ]
  }
}


In [15]:
print('diagnosis_environment:')
show_json(diagnosis_environment())
print('\ncoach_environment:')
show_json(diagnose_environment())


diagnosis_environment:
{
  "diagnosis_agent_version": "2026-06-17-fixed-problem-diagnosis",
  "az_path": "/Users/xumuchi/.local/bin:/Users/xumuchi/.pyenv/shims:/opt/homebrew/bin:/opt/anaconda3/bin:/opt/anaconda3/condabin:/Library/Frameworks/Python.framework/Versions/3.10/bin:/opt/homebrew/bin:/opt/homebrew/sbin:/Library/Frameworks/Python.framework/Versions/3.14/bin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/Library/TeX/texbin:/Users/xumuchi/.orbstack/bin",
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}

coach_environment:
{
  "coach_agent_version": "2026-06-16-structured-output-check",
  "az_pa

## 2. 先做诊断

这里直接拿题目里的 `student_answer` 当第一次诊断输入。

In [16]:
diagnosis_agent = FoundryDiagnosisAgent(
    model_deployment=DIAGNOSIS_MODEL,
    use_default_credential=USE_DEFAULT_CREDENTIAL,
    use_ai_diagnoser=USE_AI_DIAGNOSER,
)

diagnosis_session = diagnosis_agent.create_session(
    problem_text=QUESTION['stem'],
    reference_answer=QUESTION['solution_text'],
    student_profile=STUDENT_PROFILE,
    coach_max_turns=COACH_MAX_TURNS,
)

diagnosis_result = diagnosis_agent.diagnose(
    QUESTION['student_answer'],
    session=diagnosis_session,
)

show_json(diagnosis_result.as_dict())


{
  "problem_text": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
  "reference_answer": "由 a_{n+1}=3a_n+1，得 a_{n+1}+1/2=3a_n+1+1/2=3(a_n+1/2)。又 a_1+1/2=1/2+1/2=1，因此新数列满足后一项等于前一项乘 3，且首项为 1，所以 {a_n+1/2} 是首项为 1、公比为 3 的等比数列。",
  "student_answer": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。",
  "student_profile": "学生会从递推式入手，但经常不会主动构造中间量，证明题收尾也不太稳。",
  "error_type": "concept_gap",
  "confidence": 0.96,
  "reason": "学生不明白需要通过构造 a_n+1/2 来把递推式化为等比形式，说明对概念和技巧缺乏理解。",
  "evidence": "学生回答中写到‘不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列’",
  "coach_mode": "direct_explain",
  "coach_trap": "学生缺少必要概念，继续追问会放大挫败。",
  "coach_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
  "source": "ai_tool_text_json_validated"
}


In [17]:
diagnosis_event = diagnosis_result_to_memory_event(
    diagnosis_result.as_dict(),
    question_id=QUESTION_ID,
    binding=BINDING_CONTEXT,
    occurred_at=BASE_TIME.isoformat(),
    source_name=QUESTION.get('source_name'),
    source_section=QUESTION.get('source_section'),
)

show_json(diagnosis_event)


{
  "occurred_at": "2026-06-23T19:00:00+08:00",
  "question_id": "wq_72b68db02925",
  "error_type": "concept_gap",
  "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
  "secondary_node_ids": [],
  "reason": "学生不明白需要通过构造 a_n+1/2 来把递推式化为等比形式，说明对概念和技巧缺乏理解。",
  "evidence": "学生回答中写到‘不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列’",
  "confidence": 0.96,
  "source_name": "binder_notebook_demo",
  "source_section": "数列递推转等比",
  "coach_mode": "direct_explain",
  "coach_trap": "学生缺少必要概念，继续追问会放大挫败。",
  "coach_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
  "diagnosis_source": "ai_tool_text_json_validated"
}


## 3. 做几轮引导

下面这几句学生回复你可以自由改。当前只是为了演示：

- 第一轮：学生还是说不清为什么加 `1/2`
- 第二轮：学生开始意识到要把递推式改造成等比递推
- 第三轮：学生能把证明收尾

In [18]:
FOLLOW_UP_STUDENT_REPLIES = [
    '是不是要先把原递推式改造成一个新的等比递推？但我还不知道为什么偏偏加 1/2。',
    '因为加上 1/2 以后，右边会变成 3(a_n+1/2)，这样就能把新数列写成后一项等于前一项乘 3。',
    '再结合首项 a_1+1/2=1，就能说明它是首项为 1、公比为 3 的等比数列。',
]

FOLLOW_UP_STUDENT_REPLIES


['是不是要先把原递推式改造成一个新的等比递推？但我还不知道为什么偏偏加 1/2。',
 '因为加上 1/2 以后，右边会变成 3(a_n+1/2)，这样就能把新数列写成后一项等于前一项乘 3。',
 '再结合首项 a_1+1/2=1，就能说明它是首项为 1、公比为 3 的等比数列。']

In [19]:
coach_agent = FoundryCoachAgent(
    model_deployment=COACH_MODEL,
    use_default_credential=USE_DEFAULT_CREDENTIAL,
    use_ai_reply_analyzer=USE_AI_REPLY_ANALYZER,
)

coach_session = diagnosis_result.build_coach_session(
    coach_agent,
    student_profile=STUDENT_PROFILE,
    max_turns=COACH_MAX_TURNS,
    initial_student_reply=QUESTION['student_answer'],
)

coach_turns = []

first_response = coach_agent.start(coach_session)
coach_turns.append({
    'student_reply': QUESTION['student_answer'],
    'coach_response': first_response.as_dict(),
})

print('=== coach turn 1 ===')
show_json(coach_turns[-1])

for student_reply in FOLLOW_UP_STUDENT_REPLIES:
    if coach_session.done:
        break
    response = coach_agent.reply(student_reply, session=coach_session)
    coach_turns.append({
        'student_reply': student_reply,
        'coach_response': response.as_dict(),
    })
    print(f'\n=== coach turn {len(coach_turns)} ===')
    show_json(coach_turns[-1])


=== coach turn 1 ===
{
  "student_reply": "我知道要从递推式入手，但不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列。",
  "coach_response": {
    "content": "别担心，这一步其实很直接。我们把 a_n 加上 1/2，记作 b_n = a_n+½，代进去会得到 b_{n+1}=3·b_n，这说明每一项都是前一项的 3 倍，所以 {b_n} 是等比数列。你先算算 b₁ 和 b₂，看看它们的比是多少，验证一下这个结论吧。",
    "reply_quality": "empty",
    "strategy": {
      "mode": "direct_explain",
      "trap": "学生缺少必要概念，继续追问会放大挫败。",
      "prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。"
    },
    "strategy_mode": "direct_explain",
    "strategy_trap": "学生缺少必要概念，继续追问会放大挫败。",
    "strategy_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
    "turn_index": 1,
    "done": false,
    "stop_reason": "continue",
    "reply_analysis": {
      "quality": "empty",
      "understands": false,
      "completed": false,
      "reason": "模型输出不是可解析的 JSON，已使用本地 fallback。",
      "source": "fallback_heuristic"
    }
  }
}

=== coach turn 2 ===
{
  "student_reply": "是不是要先把原递推式改造成一个新的等比递推？但我还不知道为什么偏偏加 1/2。",
  "coach_response": {
    "content": "因为递推式里有个 “+1”，如果我们把每一

In [20]:
coach_events = []

for index, turn in enumerate(coach_turns, start=1):
    event = coach_response_to_memory_event(
        turn['coach_response'],
        question_id=QUESTION_ID,
        binding=BINDING_CONTEXT,
        occurred_at=(BASE_TIME + timedelta(minutes=2 * index)).isoformat(),
        error_type=diagnosis_result.error_type.value,
        source_name=QUESTION.get('source_name'),
        source_section=QUESTION.get('source_section'),
    )
    coach_events.append(event)

show_json(coach_events)


[
  {
    "occurred_at": "2026-06-23T19:02:00+08:00",
    "question_id": "wq_72b68db02925",
    "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
    "secondary_node_ids": [],
    "error_type": "concept_gap",
    "coach_mode": "direct_explain",
    "coach_trap": "学生缺少必要概念，继续追问会放大挫败。",
    "coach_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
    "reply_quality": "empty",
    "understands": false,
    "completed": false,
    "reason": "模型输出不是可解析的 JSON，已使用本地 fallback。",
    "turn_index": 1,
    "done": false,
    "stop_reason": "continue",
    "source_name": "binder_notebook_demo",
    "source_section": "数列递推转等比"
  },
  {
    "occurred_at": "2026-06-23T19:04:00+08:00",
    "question_id": "wq_72b68db02925",
    "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
    "secondary_node_ids": [],
    "error_type": "concept_gap",
    "coach_mode": "direct_explain",
    "coach_trap": "学生缺少必要概念，继续追问会放大挫败。",
    "coach_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
    "reply_quality": "empty",


## 4. 生成 student_memory_profile

这里先只喂：

- 1 条诊断事件
- 若干条 coach 引导事件

后面如果你要，也可以再把 `review_state` 或 `review_events` 一起加进去。

In [21]:
student_memory_profile = build_profile(
    student_id=STUDENT_ID,
    diagnosis_events=[diagnosis_event],
    coach_events=coach_events,
)

PROFILE_PATH = SCRATCH_DIR / 'student_memory_profile_demo.json'
DIAGNOSIS_EVENTS_PATH = SCRATCH_DIR / 'diagnosis_events_demo.json'
COACH_EVENTS_PATH = SCRATCH_DIR / 'coach_events_demo.json'

PROFILE_PATH.write_text(
    json.dumps(student_memory_profile, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
DIAGNOSIS_EVENTS_PATH.write_text(
    json.dumps([diagnosis_event], ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
COACH_EVENTS_PATH.write_text(
    json.dumps(coach_events, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

print('已写出：')
print(PROFILE_PATH)
print(DIAGNOSIS_EVENTS_PATH)
print(COACH_EVENTS_PATH)


已写出：
/Users/xumuchi/Desktop/TeachAgent/scratch/diagnosis_coach_memory_demo/student_memory_profile_demo.json
/Users/xumuchi/Desktop/TeachAgent/scratch/diagnosis_coach_memory_demo/diagnosis_events_demo.json
/Users/xumuchi/Desktop/TeachAgent/scratch/diagnosis_coach_memory_demo/coach_events_demo.json


In [22]:
summary = student_memory_profile['personalization_summary']
compact_view = {
    'student_id': student_memory_profile['student_id'],
    'dominant_error_type': summary.get('dominant_error_type'),
    'recommended_teaching_mode': summary.get('recommended_teaching_mode'),
    'recommended_review_mode': summary.get('recommended_review_mode'),
    'top_recurrent_nodes': summary.get('top_recurrent_nodes', []),
    'event_history': student_memory_profile.get('event_history', []),
}

show_json(compact_view)


{
  "student_id": "demo_student",
  "dominant_error_type": "concept_gap",
  "recommended_teaching_mode": "concept_first",
  "recommended_review_mode": "leaf_first",
  "top_recurrent_nodes": [
    {
      "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
      "title": "等比数列通项公式",
      "dominant_error_type": "concept_gap",
      "recommended_intervention": "reteach_concept"
    }
  ],
  "event_history": [
    {
      "event_type": "diagnosis",
      "occurred_at": "2026-06-23T19:00:00+08:00",
      "question_id": "wq_72b68db02925",
      "error_type": "concept_gap",
      "primary_node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
      "secondary_node_ids": [],
      "reason": "学生不明白需要通过构造 a_n+1/2 来把递推式化为等比形式，说明对概念和技巧缺乏理解。",
      "evidence": "学生回答中写到‘不知道为什么要构造 a_n+1/2，也不会判断新数列为什么是等比数列’",
      "confidence": 0.96,
      "coach_mode": "direct_explain",
      "coach_trap": "学生缺少必要概念，继续追问会放大挫败。",
      "coach_prompt": "学生仍然没思路。直接讲清当前关键步骤，再立刻给一个极短验证问题。",
      "diagnosis_source": null
    },
    {


In [23]:
show_json(student_memory_profile)


{
  "record_id": "student_memory.demo_student",
  "student_id": "demo_student",
  "profile_version": "student_memory_profile_v1",
  "generated_at": "2026-06-23T16:01:06.989213+00:00",
  "updated_at": "2026-06-23T19:06:00+08:00",
  "error_type_counts": {
    "concept_gap": 1,
    "missing_strategy": 0,
    "misreading": 0,
    "calculation": 0,
    "careless": 0
  },
  "node_memories": [
    {
      "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
      "linked_question_ids": [
        "wq_72b68db02925"
      ],
      "error_type_counts": {
        "concept_gap": 1,
        "missing_strategy": 0,
        "misreading": 0,
        "calculation": 0,
        "careless": 0
      },
      "diagnosis_count": 1,
      "observed_wrong_count": 1,
      "review_wrong_count": 0,
      "review_correct_count": 0,
      "review_partial_count": 0,
      "practice_request_count": 0,
      "consecutive_wrong_count": 1,
      "mastery_hint": 0.0,
      "stability_hint": 0.0,
      "last_seen_at": "2026-06-23

## 5. 你之后可以怎么改

最常改的地方有 3 个：

1. 改 `FOLLOW_UP_STUDENT_REPLIES`
   - 看不同学生回复会不会改变 coach 和 memory 结果

2. 换题目 / 换绑定结果
   - 直接换成别的 `wrong_binder` 产物即可

3. 在 `build_profile(...)` 里继续加入 `review_state` 或 `review_events`
   - 这样第四阶段就会更接近你完整系统里的长期学生画像